In [1]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Image transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
])

# Load dataset
train_data = datasets.ImageFolder("train/", transform=transform)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

# Check classes
print(train_data.classes)

['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal']


In [4]:
from torch.utils.data import random_split

# Split 80% train, 20% validation
train_size = int(0.8 * len(train_data))
val_size = len(train_data) - train_size

train_dataset, val_dataset = random_split(train_data, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [5]:
import torchvision.models as models
import torch.nn as nn

model = models.efficientnet_b0(pretrained=True)

# Modify final layer for 4 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)

C:\Users\dev\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\dev\AppData\Local\Programs\Python\Python313\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\dev/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100.0%


In [6]:
import torchvision.models as models
import torch.nn as nn

model = models.efficientnet_b0(pretrained=True)

# Modify final layer for 4 classes
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 4)

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

print(device)

cpu


In [8]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [9]:
epochs = 5

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}, Accuracy: {100 * correct / total:.2f}%")

Epoch 1, Loss: 52.0479, Accuracy: 81.59%
Epoch 2, Loss: 29.8790, Accuracy: 89.48%
Epoch 3, Loss: 26.9644, Accuracy: 90.42%
Epoch 4, Loss: 23.5165, Accuracy: 91.49%
Epoch 5, Loss: 18.9291, Accuracy: 93.30%


In [12]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {100 * correct / total:.2f}%")

Validation Accuracy: 94.31%


In [17]:
import os
import pandas as pd
from PIL import Image

test_path = "test/"
results = []

model.eval()

for img_name in os.listdir(test_path):
    # ✅ Only allow image files
    if not img_name.lower().endswith((".jpg", ".jpeg", ".png")):
        continue

    img_path = os.path.join(test_path, img_name)

    image = Image.open(img_path).convert("RGB")
    image = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(image)
        _, predicted = torch.max(output, 1)

    results.append([img_name, predicted.item()])

df = pd.DataFrame(results, columns=["image_id", "label"])
df.to_csv("solution.csv", index=False)

print("✅ solution.csv created!")

✅ solution.csv created!


In [18]:
import torch

torch.save(model.state_dict(), "model.pth")

print("✅ Model saved as model.pth")

✅ Model saved as model.pth
